# MiniMax H3 — 静止画1枚から縦動画

G4 は生成のときだけ使う。重みと LoRA のダウンロードは CPU ランタイム。

- Drive は **fireworker06@gmail.com**。重みは `マイドライブ/h3-weights/MiniMax-H3`。LoRA は `マイドライブ/h3-weights/loras/`。完成 mp4 は `マイドライブ/h3-runner/output/orbis01.mp4`。
- 既定は FL2VA が2本（6秒と9秒）。静止画は `image=` の最初のコマ。steps は 9（終点 0 を含むので評価は 8 回）。video shift 6、audio shift 3。
- bf16 では LoRA を融合してから offload する。融合しないと静止画1枚あたりおおよそ 9〜13% 増える。int8 では LoRA を無効にする。
- Turbo は Hugging Face `lightx2v/Minimax-h3-Turbo` の `fl2v_turbo_8step_v1.0_768p`（ComfyUI bf16）。Combat BASE V2 は Hugging Face `JOKER141/MiniMax-H3-Combat-Base-V2` の `H3_Combat_V2.safetensors`（**BUNNY（作者 FourBunny）**）。Motion Continuity Repair V2 は Hugging Face `JOKER141/MiniMax-H3-General-Motion-Continuity-Repair` の `Motion_Repair_V2.safetensors`（カード名は BUNNY Motion Repair V2、配布は JOKER141）。使うときはそのクレジットを書く。アダルト用ノートはこのブランチに無い。
- AdaLN の入力幅が 8（Pruned 学習）の LoRA は拒否する。
- 既定が置くベースは **144.1GB**（transformer。`transformer_ref` は Ref2VA のときだけ）。`FL2VA/` と `Ref2VA/`（各 144.1GB）は落とさない。
- `/content/drive` の空き表示は VM ディスクの値なので、そこで止めない。シャードを1本ずつ `/content/tmp_hf` に落として Drive へ移す。
- 既存の `orbis01_6s.mp4` は上書きしない。比較の A に使う。新しい前半は `orbis01_fl2va_6s.mp4`。
- `HF_TOKEN` はライセンス同意が要るときだけ。Turbo、Combat、Repair はどれも Hugging Face。値は表示しない。

[Open in Colab](https://colab.research.google.com/github/fireworker011/Research/blob/cursor/h3-sfw-colab-6dc5/h3-runner/minimax_h3_still.ipynb)


## A. CPU ランタイム

ランタイムのタイプを **CPU**（GPU なし）にする。上の導入セルの次から、A のセルだけ実行する。


### A. Drive

fireworker06@gmail.com でマウントする。空きの数字は参考。Colab のマウントは VM ディスクの値を返すので Drive 実容量ではない。


In [ ]:
import os
import shutil
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive")
WEIGHTS = Path("/content/drive/MyDrive/h3-weights")
OUT = Path("/content/drive/MyDrive/h3-runner/output")
WEIGHTS.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
os.environ["H3_HF_CACHE"] = str(WEIGHTS)
os.environ["H3_OUT_DIR"] = str(OUT)
os.environ["HF_HOME"] = "/content/tmp_hf/hf-home"

token = ""
try:
    token = userdata.get("HF_TOKEN") or ""
except Exception:
    token = ""
if token:
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    print("HF_TOKEN をシークレットから読みました（値は表示しません）")
else:
    print("HF_TOKEN は未設定。401 のときだけ、ライセンス同意後にシークレットへ入れてこのセルからやり直す。")

usage = shutil.disk_usage("/content/drive")
root = shutil.disk_usage("/")
print(f"参考: /content/drive の statvfs 空き {usage.free / 1e9:.1f} GB / 全体 {usage.total / 1e9:.1f} GB（10^9）")
print("Colab のマウントは VM ディスクの値を返すので Drive 実容量ではない。この数字では止めない。")
print(f"ローカル / の空き {root.free / 1e9:.1f} GB。drivefs のキャッシュはここを使う。")
print("両方置く合計は 210.3GB。シャードを1本ずつ落とす。")
print("重み:", WEIGHTS)
print("完成 mp4:", OUT)
print("画面のアカウントが fireworker06@gmail.com であることを確認する。")


### A. リポジトリ

Research を `/content/Research` に clone する。ランナーは `/content/Research/h3-runner/run_h3.py`。シンボリックリンクは使わない。


In [ ]:
import subprocess
from pathlib import Path

def clone_runner():
    root = Path("/content/Research")
    branch = "cursor/h3-sfw-colab-6dc5"
    repo = "https://github.com/fireworker011/Research.git"
    script = root / "h3-runner" / "run_h3.py"
    if not script.is_file():
        if root.exists():
            raise SystemExit(f"{root} があるが {script} が無い。このフォルダを消してやり直す。")
        subprocess.check_call(["git", "clone", "--depth", "1", "--branch", branch, repo, str(root)])
    else:
        subprocess.check_call(["git", "-C", str(root), "fetch", "--depth", "1", "origin", branch])
        subprocess.check_call(["git", "-C", str(root), "checkout", branch])
        subprocess.check_call(["git", "-C", str(root), "pull", "--ff-only", "origin", branch])
    if not script.is_file():
        raise SystemExit(f"ランナーが無い: {script}")
    print("runner", script)
    return script

clone_runner()


### A. パッケージ

ダウンロードに `huggingface_hub` だけ入れる。torch は触らない。


In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.25.0"])
print("huggingface_hub ok")


### A. 重み準備

無ければシャードを1本ずつ Drive に落とす。サイズが一致するファイルは飛ばして続きからやる。GPU は使わない。

そのあと LoRA を `マイドライブ/h3-weights/loras/` に置く。既にあるファイルは飛ばす。

- Turbo: Hugging Face `lightx2v/Minimax-h3-Turbo` の `minimax_h3_fl2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors`
- Motion Continuity Repair V2: Hugging Face `JOKER141/MiniMax-H3-General-Motion-Continuity-Repair` の `Motion_Repair_V2.safetensors`（カード名は BUNNY Motion Repair V2。V1 の `Motion_Repair.safetensors` は置かない）
- Combat BASE V2: Hugging Face `JOKER141/MiniMax-H3-Combat-Base-V2` の `H3_Combat_V2.safetensors`。別名のファイルは置かない。

Repair のヘッダは ai-toolkit（`diffusion_model.` + `lora_A`/`lora_B`、AdaLN キーなし）。AdaLN 入力幅 8 は拒否する。diffusers の `load_lora_weights` がこの形を変換するので、別ファイルへの事前変換はしない。

一時ファイルは `/content/tmp_hf`。1本ごとに Drive へ移してローカルを消し、`os.sync()` する。`drive.flush_and_unmount` は使わない。`/` の空きが 30GB を切ったらキャッシュが上がるまで待つ。

`--vram-gb 95 --host-ram-gb 176.9` は前回の G4 実測で、計画の文面用。このセルはディスクだけ見る。


In [ ]:
import os
import subprocess
import sys
import threading
from pathlib import Path

def run_logged(cmd):
    """Colab does not show a child process's own fds. Print them from this cell."""
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    stderr_lines = []

    def pump(stream, acc):
        for line in stream:
            print(line, end="", flush=True)
            if acc is not None:
                acc.append(line)

    out_thread = threading.Thread(target=pump, args=(proc.stdout, None))
    err_thread = threading.Thread(target=pump, args=(proc.stderr, stderr_lines))
    out_thread.start()
    err_thread.start()
    code = proc.wait()
    out_thread.join()
    err_thread.join()
    if code != 0:
        print(f"\n終了コード {code}。stderr 全文:", flush=True)
        print("".join(stderr_lines), flush=True)
        raise SystemExit(code)
    return code

def clone_runner():
    root = Path("/content/Research")
    branch = "cursor/h3-sfw-colab-6dc5"
    repo = "https://github.com/fireworker011/Research.git"
    script = root / "h3-runner" / "run_h3.py"
    if not script.is_file():
        if root.exists():
            raise SystemExit(f"{root} があるが {script} が無い。このフォルダを消してやり直す。")
        subprocess.check_call(["git", "clone", "--depth", "1", "--branch", branch, repo, str(root)])
    else:
        subprocess.check_call(["git", "-C", str(root), "fetch", "--depth", "1", "origin", branch])
        subprocess.check_call(["git", "-C", str(root), "checkout", branch])
        subprocess.check_call(["git", "-C", str(root), "pull", "--ff-only", "origin", branch])
    if not script.is_file():
        raise SystemExit(f"ランナーが無い: {script}")
    print("runner", script)
    return script

script = clone_runner()
cmd = [
    sys.executable, "-u", str(script),
    "--preset", "orbis01",
    "--cache-dir", os.environ["H3_HF_CACHE"],
    "--out-dir", os.environ["H3_OUT_DIR"],
    "--prepare-weights",
    "--vram-gb", "95",
    "--host-ram-gb", "176.9",
]
run_logged(cmd)
import sys

sys.path.insert(0, "/content/Research/h3-runner")
from h3_runner.loras import prepare_fast_loras

lora_dir = Path(os.environ["H3_HF_CACHE"]) / "loras"
prepare_fast_loras(lora_dir)
print("LoRA:", lora_dir)
print("重み準備が終わった。ランタイムを G4 に変えて B へ進む。")


## B. G4 ランタイム

ランタイムのタイプを **G4 GPU**（RTX PRO 6000）にする。CPU とは別の VM なので、Drive とリポジトリはここでも取る。生成はこのあとの1セル。


### B. 準備

Drive、パッケージ、リポジトリ。`pip install -U torchao==0.18.0` で `FqnToConfig` を直す。Colab の torch は置き換えない。


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive")
WEIGHTS = Path("/content/drive/MyDrive/h3-weights")
OUT = Path("/content/drive/MyDrive/h3-runner/output")
if not WEIGHTS.is_dir():
    raise SystemExit(f"重みフォルダが無い: {WEIGHTS}。先に CPU の A を実行する。")
OUT.mkdir(parents=True, exist_ok=True)
os.environ["H3_HF_CACHE"] = str(WEIGHTS)
os.environ["H3_OUT_DIR"] = str(OUT)
os.environ["HF_HOME"] = str(WEIGHTS / "hf-home")

token = ""
try:
    token = userdata.get("HF_TOKEN") or ""
except Exception:
    token = ""
if token:
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    print("HF_TOKEN をシークレットから読みました（値は表示しません）")

usage = shutil.disk_usage("/content/drive")
print(f"Drive 空き {usage.free / 1e9:.1f} GB / 全体 {usage.total / 1e9:.1f} GB（10^9）")

import torch
if not torch.cuda.is_available():
    raise SystemExit("GPU がオフです。ランタイムを G4 にして、このセルからやり直す。")
props = torch.cuda.get_device_properties(0)
vram = props.total_memory / 1024 ** 3
mem = Path("/proc/meminfo").read_text(encoding="utf-8")
host = int(next(line.split()[1] for line in mem.splitlines() if line.startswith("MemTotal:"))) / 1024 ** 2
print(f"GPU: {props.name}  VRAM: {vram:.1f} GB  ホストRAM: {host:.1f} GB  torch {torch.__version__}")
os.environ["H3_VRAM_GB"] = f"{vram:.4f}"
os.environ["H3_HOST_RAM_GB"] = f"{host:.4f}"
if vram < 24:
    raise SystemExit("VRAM が 24GB 未満です。G4 に変えてやり直す。")

if shutil.which("ffmpeg") is None:
    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "ffmpeg"])

pkgs = [
    "git+https://github.com/huggingface/diffusers.git@5ff8e59ff9fe81c6e2df4fb4c6ea0d97a5df5ab2",
    "transformers>=4.45.0",
    "accelerate>=0.34.0",
    "safetensors>=0.4.3",
    "huggingface_hub>=0.25.0",
    "av>=11.0.0",
    "imageio>=2.34.0",
    "imageio-ffmpeg>=0.5.0",
    "soundfile>=0.12.0",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed", *pkgs])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "torchao==0.18.0"])
from torchao.quantization import FqnToConfig
print("torchao FqnToConfig ok", "ffmpeg", shutil.which("ffmpeg"))

def clone_runner():
    root = Path("/content/Research")
    branch = "cursor/h3-sfw-colab-6dc5"
    repo = "https://github.com/fireworker011/Research.git"
    script = root / "h3-runner" / "run_h3.py"
    if not script.is_file():
        if root.exists():
            raise SystemExit(f"{root} があるが {script} が無い。このフォルダを消してやり直す。")
        subprocess.check_call(["git", "clone", "--depth", "1", "--branch", branch, repo, str(root)])
    else:
        subprocess.check_call(["git", "-C", str(root), "fetch", "--depth", "1", "origin", branch])
        subprocess.check_call(["git", "-C", str(root), "checkout", branch])
        subprocess.check_call(["git", "-C", str(root), "pull", "--ff-only", "origin", branch])
    if not script.is_file():
        raise SystemExit(f"ランナーが無い: {script}")
    print("runner", script)
    return script

clone_runner()


### B. 生成

この1セルが生成の最初から結合まで。ダウンロードしない。Drive の `h3-weights` を直接読む。

FL2VA 6秒+9秒、steps 9、video shift 6、Turbo 強さ 1.0。`orbis01_fl2va_6s.mp4` が Drive にあればその本をスキップする。既存の `orbis01_6s.mp4` は触らない。

切れたら、ランタイムが生きていればこのセルだけ、死んでいれば「B. 準備」からやり直す。


In [ ]:
import os
import subprocess
import sys
import threading
from pathlib import Path

def run_logged(cmd):
    """Colab does not show a child process's own fds. Print them from this cell."""
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    stderr_lines = []

    def pump(stream, acc):
        for line in stream:
            print(line, end="", flush=True)
            if acc is not None:
                acc.append(line)

    out_thread = threading.Thread(target=pump, args=(proc.stdout, None))
    err_thread = threading.Thread(target=pump, args=(proc.stderr, stderr_lines))
    out_thread.start()
    err_thread.start()
    code = proc.wait()
    out_thread.join()
    err_thread.join()
    if code != 0:
        print(f"\n終了コード {code}。stderr 全文:", flush=True)
        print("".join(stderr_lines), flush=True)
        raise SystemExit(code)
    return code

script = Path("/content/Research/h3-runner/run_h3.py")
if not script.is_file():
    raise SystemExit(f"ランナーが無い: {script}。B. 準備からやり直す。")
cmd = [
    sys.executable, "-u", str(script),
    "--preset", "orbis01",
    "--cache-dir", os.environ["H3_HF_CACHE"],
    "--out-dir", os.environ["H3_OUT_DIR"],
    "--vram-gb", os.environ["H3_VRAM_GB"],
    "--host-ram-gb", os.environ["H3_HOST_RAM_GB"],
    "--seed", "0",
]

lora = Path(os.environ["H3_HF_CACHE"]) / "loras" / "minimax_h3_fl2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors"
if not lora.is_file():
    raise SystemExit(f"Turbo LoRA が無い: {lora}。CPU の「A. 重み準備」を先に実行する。")
cmd.extend(["--steps", "9", "--video-shift", "6", "--lora", f"{lora}:1.0"])
run_logged(cmd)
out = Path(os.environ["H3_OUT_DIR"]) / "orbis01.mp4"
if not out.is_file():
    raise SystemExit(f"完成 mp4 がありません: {out}")
print("完成:", out, "bytes", out.stat().st_size)


### B. 比較テスト

この1セルだけ。G4 で、6秒・768×1344・seed 0・同じプロンプト（`orbis01_t2va_6s.txt`）と同じ静止画（`sakura-ref.jpg`）を順に出す。

- A: 既存の `orbis01_6s.mp4`。このセルは作らない。
- B: FL2VA + Turbo 1.0 → `test_b_fl2va_turbo.mp4`
- C: B + Combat BASE V2 0.7 → `test_c_turbo_combat.mp4`（Combat のファイルが無いときは飛ばす）
- D: C + Motion Continuity Repair V2 0.6 → `test_d_turbo_combat_repair.mp4`（Combat が無いときは飛ばす）
- D': B + Repair 0.6 → `test_dp_turbo_repair.mp4`（Combat が無くても回す）

保存先は `マイドライブ/h3-runner/output/`。各段（読み込み、テキストエンコード、ノイズ除去、VAE）の秒数を表にする。1本が落ちても次へ進む。Combat は BUNNY（作者 FourBunny）。Repair は BUNNY Motion Repair V2（配布 JOKER141）。クレジットを書く。


In [ ]:
import os
import re
import subprocess
import sys
import threading
from pathlib import Path

def run_capture(cmd):
    """Print the child live, and keep the text so one failure can continue."""
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    chunks = []

    def pump(stream):
        for line in stream:
            print(line, end="", flush=True)
            chunks.append(line)

    out_thread = threading.Thread(target=pump, args=(proc.stdout,))
    err_thread = threading.Thread(target=pump, args=(proc.stderr,))
    out_thread.start()
    err_thread.start()
    code = proc.wait()
    out_thread.join()
    err_thread.join()
    return code, "".join(chunks)

script = Path("/content/Research/h3-runner/run_h3.py")
prompt = Path("/content/Research/h3-runner/prompts/orbis01_t2va_6s.txt")
still = Path("/content/Research/h3-runner/assets/sakura-ref.jpg")
if not script.is_file():
    raise SystemExit(f"ランナーが無い: {script}。B. 準備からやり直す。")
lora_dir = Path(os.environ["H3_HF_CACHE"]) / "loras"
turbo = lora_dir / "minimax_h3_fl2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors"
combat = lora_dir / "H3_Combat_V2.safetensors"
repair = lora_dir / "Motion_Repair_V2.safetensors"
out_dir = Path(os.environ["H3_OUT_DIR"])
baseline = out_dir / "orbis01_6s.mp4"
print("A 基準:", baseline, "bytes", baseline.stat().st_size if baseline.is_file() else "無い")
print("プロンプト:", prompt)
print("静止画:", still)
print("Combat は BUNNY（作者 FourBunny）。Repair は BUNNY Motion Repair V2（配布 JOKER141）。動画にクレジットを書く。")

variants = [
    ("B", [f"{turbo}:1.0"], out_dir / "test_b_fl2va_turbo.mp4"),
]
if combat.is_file():
    variants.append(("C", [f"{turbo}:1.0", f"{combat}:0.7"], out_dir / "test_c_turbo_combat.mp4"))
    variants.append(("D", [f"{turbo}:1.0", f"{combat}:0.7", f"{repair}:0.6"], out_dir / "test_d_turbo_combat_repair.mp4"))
else:
    print("H3_Combat_V2.safetensors が無いので C と D は飛ばす。A. 重み準備からやり直す。")
variants.append(("D'", [f"{turbo}:1.0", f"{repair}:0.6"], out_dir / "test_dp_turbo_repair.mp4"))
rows = {"A": {"load": "—", "text": "—", "denoise": "—", "vae": "—", "out": baseline.name if baseline.is_file() else "無い"}}
for name, loras, dest in variants:
    missing = [Path(item.rsplit(":", 1)[0]) for item in loras if not Path(item.rsplit(":", 1)[0]).is_file()]
    if missing:
        rows[name] = {"load": "失敗", "text": "失敗", "denoise": "失敗", "vae": "失敗", "out": "LoRA が無い: " + ", ".join(path.name for path in missing)}
        print(f"{name} を飛ばす。LoRA が無い:", ", ".join(str(path) for path in missing))
        continue
    cmd = [
        sys.executable, "-u", str(script),
        "--task", "fl2va",
        "--prompt-file", str(prompt),
        "--image", str(still),
        "--duration", "6",
        "--width", "768",
        "--height", "1344",
        "--seed", "0",
        "--steps", "9",
        "--video-shift", "6",
        "--offload", "bf16",
        "--cache-dir", os.environ["H3_HF_CACHE"],
        "--vram-gb", os.environ.get("H3_VRAM_GB", "95"),
        "--host-ram-gb", os.environ.get("H3_HOST_RAM_GB", "176.9"),
        "--out", str(dest),
    ]
    for spec in loras:
        cmd.extend(["--lora", spec])
    print(f"\n==== {name} → {dest.name} ====", flush=True)
    code, text = run_capture(cmd)
    found = re.search(r"TIMING load=([0-9.]+) text=([0-9.]+) denoise=([0-9.]+) vae=([0-9.]+)", text)
    if code == 0 and found and dest.is_file():
        rows[name] = {
            "load": found.group(1),
            "text": found.group(2),
            "denoise": found.group(3),
            "vae": found.group(4),
            "out": f"{dest.name} ({dest.stat().st_size} bytes)",
        }
    else:
        tail = text.strip().splitlines()[-3:]
        rows[name] = {
            "load": "失敗",
            "text": "失敗",
            "denoise": "失敗",
            "vae": "失敗",
            "out": f"終了コード {code}: " + " / ".join(tail),
        }
        print(f"{name} は落ちた。次へ進む。", flush=True)

skipped = {"load": "飛ばした", "text": "飛ばした", "denoise": "飛ばした", "vae": "飛ばした", "out": "Combat が無い"}
rows.setdefault("C", skipped)
rows.setdefault("D", skipped)
print("\n| 段 | A 基準 | B Turbo | C +Combat 0.7 | D C+Repair 0.6 | D' B+Repair 0.6 |")
print("|---|---|---|---|---|---|")
labels = [("読み込み", "load"), ("テキストエンコード", "text"), ("ノイズ除去", "denoise"), ("VAE", "vae"), ("出力", "out")]
order = ["A", "B", "C", "D", "D'"]
for label, key in labels:
    print("| " + " | ".join([label] + [rows[name][key] for name in order]) + " |")
failed = [name for name in ("B", "C", "D", "D'") if rows[name]["load"] == "失敗"]
if failed:
    raise SystemExit("落ちた段: " + ", ".join(failed) + "。表は上。")
